In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install ultralytics

In [ ]:


!pip install -q --upgrade ultralytics

import os
import yaml
import torch
import torch.nn as nn
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules as ul_modules

class ViTBlock(nn.Module):
    def __init__(self, num_heads=4):
        super().__init__()
        self.num_heads = num_heads

        self.qkv = None
        self.proj = None
        self.bn = None
        self.mlp = None

    def build(self, c):
        heads = min(self.num_heads, c)
        while c % heads != 0 and heads > 1:
            heads -= 1

        self.heads = heads
        self.head_dim = c // heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Conv2d(c, c * 3, 1, bias=False)
        self.proj = nn.Conv2d(c, c, 1, bias=False)
        self.bn = nn.BatchNorm2d(c)

        self.mlp = nn.Sequential(
            nn.Conv2d(c, c * 4, 1),
            nn.GELU(),
            nn.Conv2d(c * 4, c, 1)
        )

    def forward(self, x):
        if self.qkv is None:
            self.build(x.shape[1])  # dynamic init

        B, C, H, W = x.shape
        N = H * W

        qkv = self.qkv(x).reshape(B, 3, self.heads, self.head_dim, N)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]

        q = q.permute(0, 1, 3, 2)
        k = k.permute(0, 1, 3, 2)
        v = v.permute(0, 1, 3, 2)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        out = attn @ v
        out = out.permute(0, 1, 3, 2).reshape(B, C, H, W)

   
        out = self.bn(self.proj(out))
        out = out + self.mlp(out)

        return out + x


tasks.ViTBlock = ViTBlock
ul_modules.ViTBlock = ViTBlock

print(" ViTBlock registered")

data_path = '/kaggle/input/datasets/ramgolla17042006/cplid-updated'

classes = ["insulator", "defect"]
nc = len(classes)

data_yaml = {
    'path': data_path,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test' if os.path.exists(os.path.join(data_path, 'images/test')) else 'images/val',
    'nc': nc,
    'names': classes
}

with open("data.yaml", "w") as f:
    yaml.dump(data_yaml, f)

print(" data.yaml created")

model_yaml = f"""
nc: 2
scales:
  n: [0.33, 0.25, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C3k2, [128, True, 0.25]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C3k2, [256, True, 0.25]]
  - [-1, 1, ViTBlock, []]

  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C3k2, [512, True, 0.5]]
  - [-1, 1, ViTBlock, []]

  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C3k2, [1024, True]]
  - [-1, 1, ViTBlock, []]

  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 3, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 3, C3k2, [256, False]]   # P3

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C3k2, [512, False]]   # P4

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C3k2, [1024, False]]  # P5

  - [[18, 21, 24], 1, Detect, [2]]
"""

with open("vit_yolo.yaml", "w") as f:
    f.write(model_yaml)

print(" vit_yolo.yaml saved")

print("\nBuilding model...")
model = YOLO("vit_yolo.yaml")

try:
    model = model.load("yolo11n.pt")
    print(" Pretrained weights loaded")
except:
    print(" Training from scratch")

model.info()

print("\n Model built successfully!")

print("\nStarting training...")

results = model.train(
    data="data.yaml",
    epochs=100,
    imgsz=512,
    batch=3,
    half=True,
    device=0,
    patience=40,
    optimizer="auto",
    plots=True,
    name="insdef_pure_vit",
    pretrained=True
)

print("\nEvaluating model...")

metrics = model.val(data="data.yaml", split="test", plots=True)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"mAP@0.5       : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}")
print(f"Precision     : {metrics.box.p.mean():.4f}")
print(f"Recall        : {metrics.box.r.mean():.4f}")
print("="*60)
